# Dividend Yield (DYLD) — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
**DYLD** style factor exactly as MSCI describes it in *Barra USE4 Empirical Notes*
(Appendix A, p. 52). It is **not** a research notebook. The deliverable
is a clean, USE4-faithful DYLD factor written to parquet, suitable for
input to a multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:**
- `SHARADAR_SF1.csv` — quarterly fundamentals (ARQ dimension), `dps` field
- `SHARADAR_SEP.csv` — daily equity prices, `close` field
- `data/out/estu_monthly.parquet` — ESTU membership and market cap per signal_date

**Deliverable:** A parquet file `dyld_use4.parquet` keyed by
`(permaticker, signal_date)` containing the standardized DYLD exposure for
every stock in the coverage universe on every monthly signal date.

**Companion specs:**
- `02_style_factors/eyld/eyld_spec.ipynb` — Earnings Yield (value-adjacent; similar SF1-based construction)
- `02_style_factors/gro/gro_spec.ipynb` — Growth (also uses SF1 ARQ rolling sum pattern)

## §1. The USE4 DYLD spec — verbatim quotes

### 1a. Barra USE4 Empirical Notes, Appendix A (p. 52) — formal descriptor definition

> **Dividend Yield**
>
> *Definition:* `1.0 · DYLD`
>
> *DYLD:* *"Trailing 12-month dividends per share divided by current price per share."*

### 1b. Methodology Notes, Section 2.3 (p.9) — standardization rule

> *"Descriptors are standardized to have a mean of 0 and a standard deviation of 1.
> ... μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

---

**That is all the PDFs say about DYLD construction.**

The PDF does not specify:
- Whether "trailing 12 months" means fiscal year, calendar year, or rolling TTM
- The source of dividend data (no data vendor is named)
- How to handle zero-dividend stocks (non-payers)
- How to handle missing DPS data
- Whether outlier trimming is applied before or after standardization
- The minimum data requirement (minimum number of quarters)

## §2. End-to-end pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 0:  Setup, imports, paths                                    │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 1:  SF1 (ARQ) → trailing 12-month DPS (4-quarter sum)        │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 2:  Load ESTU (estu_monthly.parquet)                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 3:  SEP → PIT price per (permaticker, signal_date)           │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 4:  DYLD estimator  (TTM_DPS / price)                        │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 5:  Outlier treatment  (standard 3σ trim)                    │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 6:  Standardize  (cap-weighted mean=0, EW std=1)             │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 7:  Save deliverable                                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 8:  Validate                                                 │
└─────────────────────────────────────────────────────────────────────┘
```

**PIT discipline:** For any signal_date `t`, the only data permitted is data dated ≤ `t`.
- SF1: filter `datekey ≤ signal_date` (filing date, not calendardate/fiscal year end)
- SEP: use the most recent close price with `date ≤ signal_date`

**Factor type:** `fundamental` — no daily log returns, no risk-free rate, no market benchmark.

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `dyld`
**Standardized score column:** `dyld_score`

**File:** `data/out/dyld_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (used for cap-weighting) |
| `dyld` | Float64 | **Raw descriptor** — trailing 12-month DPS / current price (Stage 4 output) |
| `dyld_score` | Float64 | **Final DYLD exposure** — standardized cross-sectionally (the deliverable) |
| `n_obs` | UInt32 | Number of non-null quarterly DPS observations used in the TTM sum (1–4) |

**Coverage rules:**
- Compute `dyld` and `dyld_score` for **every stock with sufficient data**
  (`n_obs ≥ MIN_QUARTERS`, i.e. at least 1 non-null quarterly DPS), not just ESTU members.
- Zero-dividend stocks (`dyld = 0.0`) are **included** — zero is a valid exposure for non-payers.
- Standardization moments (mean, std) are computed using **ESTU members only**.
- Non-ESTU stocks get the *same* standardization parameters applied so the final
  factor is comparable across the coverage universe.
- Stocks with missing price or with price ≤ 0 are excluded (null `dyld`).

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

✅ **PDF SPEC — DYLD parameter:**
> *"Trailing 12-month dividends per share divided by current price per share."*

This implies a 12-month trailing window — implemented as 4 calendar quarters.

❓ **NOT IN PDF — Number of quarters:** The PDF says "trailing 12 months" but does not specify quarterly vs. annual aggregation or the exact lookback.

💡 **DEFAULT:** `DYLD_QUARTERS = 4` — sum the 4 most recently filed quarterly DPS values (ARQ dimension) with `datekey ≤ signal_date`. This is more timely than using an annual figure which may be stale by up to 12 months.

❓ **NOT IN PDF — Minimum data requirement:** The PDF does not specify how many quarters of DPS history are required.

💡 **DEFAULT:** `MIN_QUARTERS = 1` — require at least 1 non-null quarterly DPS filing. Zero-payers (DPS = 0.0 every quarter) satisfy this requirement and produce `dyld = 0.0`, not null.

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
FACTOR_TYPE    = "fundamental"        # no daily prices, no RF, no market benchmark
DYLD_QUARTERS  = 4                    # quarters in trailing 12-month window
MIN_QUARTERS   = 1                    # minimum non-null DPS quarters required

# 💡 DEFAULT (NOT IN PDF) — corrupt-DPS guard (two-layer filter)
# A quarterly DPS implying a single-quarter yield > 50% is a data error, not a
# real dividend. Such observations are nulled at source before the TTM sum; the
# same threshold backstops the TTM yield (dyld > 0.50 → null) in Stage 4.
DPS_Q_YIELD_MAX = 0.50

# 💡 DEFAULT (NOT IN PDF) — calendar start
CALENDAR_START = date(1999, 1, 1)
```

## §5. STAGE 1 — SF1 (ARQ) → trailing 12-month DPS

✅ **PDF SPEC:** *"Trailing 12-month dividends per share..."* — constructed from 4 most recently filed quarterly DPS values.

**Data source:** `SHARADAR_SF1.csv` filtered to `dimension = "ARQ"`, columns `(permaticker, datekey, calendardate, dps)`.

**PIT rule:** Only rows with `datekey ≤ signal_date` are visible on a given signal date. Use `datekey` (the filing date), not `calendardate` (the fiscal quarter end date), to avoid look-ahead bias.

**TTM construction:**

For each `permaticker`, sort rows by `datekey`. For each row, sum the `dps` values from that row and the 3 preceding quarterly filings:

```
ttm_dps = sum of up to 4 most recent non-null quarterly dps values (datekey order)
n_obs   = count of non-null dps values in those 4 slots
```

Implemented with `rolling_sum(window_size=4, min_samples=1).over("permaticker")` after clamping each quarter's DPS to max(dps, 0).

❓ **NOT IN PDF — Negative DPS:** Occasionally Sharadar records negative DPS due to return-of-capital adjustments or data errors.

💡 **DEFAULT:** Clamp DPS to 0 before summing — `dps_clean = max(dps, 0)`. A negative dividend makes no economic sense for this factor.

❓ **NOT IN PDF — Zero DPS:** Non-paying stocks have `dps = 0.0` in all quarters.

💡 **DEFAULT:** Retain as `ttm_dps = 0.0`. These stocks have valid DYLD = 0.

❓ **NOT IN PDF — Missing DPS (null):** Some Sharadar rows have null `dps`.

💡 **DEFAULT:** Treat null as contributing 0 to the rolling sum but do NOT count null quarters toward `n_obs`. Use `fill_null(0)` for the rolling sum, but track `n_obs` separately as a rolling count of non-null values: `rolling_sum(window_size=4, min_samples=1).over("permaticker")` on a binary `dps_not_null` indicator. If all 4 quarters are null (`n_obs = 0`), the row is excluded.

🧪 **VALIDATE:**
- SF1 ARQ rows loaded successfully; print row count and date range
- `ttm_dps ≥ 0` for all rows (no negative after clamp)
- Non-null `ttm_dps` rows for well-known dividend payers (e.g., PG, KO, JNJ) should be positive
- Non-paying tech stocks (e.g., AMZN pre-2023) should have `ttm_dps = 0.0`, not null

## §6. STAGE 2 — Load ESTU

**Identical to all other USE4 fundamental builds.**

Load `data/out/estu_monthly.parquet` — the pre-built estimation universe panel providing `(permaticker, signal_date, in_estu, mcap)` on every monthly rebalance date.

This panel drives:
1. Which `signal_date` values are in scope (the monthly calendar)
2. Which stocks are ESTU members on each date (for standardization)
3. `mcap` values for cap-weighted standardization

❓ **NOT IN PDF — ESTU boundary:** The ESTU definition is maintained in a separate build (`estu_build.ipynb`). DYLD does not modify ESTU membership.

💡 **DEFAULT:** Accept the ESTU as-is from `estu_monthly.parquet`.

🧪 **VALIDATE:**
- ESTU parquet loads without error
- ~2,500–3,500 in-ESTU stocks per date post-2005
- `mcap` is non-null and positive for all in-ESTU rows

## §7. STAGE 3 — SEP → PIT price per (permaticker, signal_date)

✅ **PDF SPEC:** *"...divided by current price per share."* — the price as of the signal date.

**Data source:** `SHARADAR_SEP.csv` — Sharadar daily equity prices. Use the `close` column.

**PIT construction:** For each `(permaticker, signal_date)` pair, take the most recent `close` with `date ≤ signal_date`. Use `join_asof` sorted by `date` within each `permaticker`.

❓ **NOT IN PDF — Price staleness:** The PDF says "current price" but does not specify how stale a price is acceptable (e.g., if a stock has not traded for 30 days).

💡 **DEFAULT:** No staleness cutoff — use whatever the most recent SEP price is, as long as `date ≤ signal_date`. Delisted or halted stocks that still appear in ESTU will have a valid price from their last trading date.

❓ **NOT IN PDF — Zero or negative price:** Occasionally SEP has zero or null prices.

💡 **DEFAULT:** Guard `price > 0` before computing DYLD. Stocks with `price ≤ 0` or null price produce null `dyld`.

🧪 **VALIDATE:**
- Price coverage: virtually all in-ESTU stocks should have a valid price
- No zero or negative prices in the final joined set
- Spot-check: a well-known large-cap (e.g., AAPL) on a known signal_date has a plausible price

## §8. STAGE 4 — DYLD estimator

✅ **PDF SPEC (Barra USE4 Empirical Notes, Appendix A, p. 52):**

> **DYLD**
>
> *"Trailing 12-month dividends per share divided by current price per share."*
>
> `DYLD = TTM_DPS / price`
>
> where `TTM_DPS` is the sum of the four most recently filed quarterly dividends per
> share (from SF1 ARQ, PIT via `datekey ≤ signal_date`) and `price` is the most
> recent daily close from SEP with `date ≤ signal_date`.

❓ **NOT IN PDF — Zero-dividend handling:** The PDF does not say whether non-paying stocks (TTM_DPS = 0) should produce DYLD = 0 or be excluded.

💡 **DEFAULT:** Retain DYLD = 0 for non-payers. Zero is economically meaningful — it is the correct exposure for a stock that pays no dividend. Excluding non-payers would bias the cross-section toward dividend payers and make the factor uninformative for the bulk of growth/tech names in the ESTU.

❓ **NOT IN PDF — Missing DPS vs. zero DPS:** Sharadar null `dps` (data not available) must be distinguished from actual zero dividend payments.

💡 **DEFAULT:** Count only non-null quarterly `dps` values toward `n_obs`. If `n_obs = 0` (all 4 quarter slots are null), produce null `dyld`. If `n_obs ≥ 1` (at least one non-null quarter, even if value is 0), produce `dyld = ttm_dps / price`. Stocks with only null `dps` history have genuinely unknown dividend status and are excluded.

❓ **NOT IN PDF — Negative DPS clamp:** Quarterly DPS can occasionally be negative in Sharadar due to return-of-capital accounting adjustments.

💡 **DEFAULT:** Clamp individual quarterly DPS to 0 before summing: `dps_clean = max(dps, 0.0)`. The TTM sum is then guaranteed non-negative.

❓ **NOT IN PDF — Minimum quarters:** The PDF says "trailing 12 months" without specifying data completeness.

💡 **DEFAULT:** `MIN_QUARTERS = 1` — require only 1 non-null quarterly filing. This maximizes coverage while excluding stocks with zero fundamental data history.

---
*The section above (PDF SPEC quote through final 💡 DEFAULT) is the `STAGE4_DESCRIPTION`
that `/build-factor` will inject verbatim into the Stage 4 stub docstring. Content below
this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation notes:**
- Stage 1 produces a TTM DPS panel: `(permaticker, datekey, ttm_dps, n_obs)` — one row per quarterly filing, sorted by datekey per ticker
- Stage 3 produces a PIT price panel: `(permaticker, signal_date, price)` — one row per (ticker, signal_date)
- Stage 4 joins these: for each `(permaticker, signal_date)`, find the most recently filed `ttm_dps` with `datekey ≤ signal_date` (via `join_asof`), then join the PIT price, then compute `dyld = ttm_dps / price`
- The `join_asof` on the SF1 TTM panel uses `by="permaticker"`, `on="datekey"`, `strategy="backward"` against the signal_date calendar

🧪 **VALIDATE:**
- Raw DYLD values should lie in [0, 0.30] for the vast majority of ESTU stocks
- Median DYLD across ESTU expected in [0.005, 0.030] (many zero-payers pull median toward low values)
- High-yield names (e.g., utilities, REITs, MLPs) should appear in Q5; zero-payers (growth tech) in Q1
- `n_obs` range 1–4; print fraction with n_obs = 4 (should be majority of established names)
- Spot-check: KO, JNJ, PG should have `dyld > 0` throughout; AMZN should have `dyld = 0` pre-2023

## §9. STAGE 5 — Outlier treatment (3σ trim)

❓ **NOT IN PDF for DYLD specifically.** Methodology Notes Section 2.2 (p.8)
describes a general outlier-treatment algorithm that applies to all descriptors:

> *"We trim these observations to three standard deviations from the mean."*

💡 **DEFAULT:** Apply 3σ trimming per signal_date, computed within ESTU using cap-weighted mean ± 3 × equal-weighted std. Applied to all stocks in the coverage universe.

**Note on REIT/MLP/Utility outliers:** High-yield sectors can produce raw DYLD values of 10–30%+. The standard 3σ trim will naturally cap these since they are statistical outliers relative to the broad ESTU. No additional hard yield cap is applied — the automated trim is sufficient.

**Lower bound note:** The lower 3σ bound `(cw_mean − 3·sigma)` will typically be negative for DYLD (since the distribution is right-skewed with a hard floor at 0). Applying `clip(lo, hi)` where `lo < 0` is safe — no DYLD values fall below `lo`, so the lower clip is never binding. Do not artificially raise `lo` to 0, as this changes the mathematical clip applied by Polars (though the effect on the data is nil since all DYLD ≥ 0).

🧪 **VALIDATE:**
- ~0.5–3% of ESTU rows should be clipped at the upper bound per date (high-yield outliers)
- Lower bound should almost never be binding (DYLD ≥ 0 by construction)
- After trimming, maximum DYLD should be well below 30%
- Cross-sectional skewness of `dyld` should decrease after trimming

## §10. STAGE 6 — Standardize (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, p.9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

**Per signal_date `t`, using only ESTU members:**

```
μ_t  = Σ_{i ∈ ESTU_t} (mcap_i · dyld_i) / Σ mcap_i   # cap-weighted mean
σ_t  = std_{i ∈ ESTU_t}(dyld_i)                        # equal-weighted std
dyld_score_{i,t} = (dyld_i − μ_t) / σ_t               # applied to ALL i
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**.
3. Cap-weighting the mean ensures the cap-weighted market portfolio has zero exposure.

**Zero-payers note:** Stocks with `dyld = 0` participate in standardization normally. After standardization they will appear at negative `dyld_score` (below the cap-weighted mean), which correctly reflects that non-payers have below-average dividend yield.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (mcap_i · dyld_score_i) / Σ mcap_i ≈ 0` (cap-weighted mean = 0 by construction)
- `std_{i ∈ ESTU}(dyld_score_i) ≈ 1` (equal-weighted std = 1 by construction)
- High-yield stocks should have positive `dyld_score`; zero-payers should have negative `dyld_score`

## §11. STAGE 7 — Save deliverable

Write the final panel to `data/out/dyld_use4.parquet`.
Compression: zstd. Statistics: True.

Include both `dyld` (raw) and `dyld_score` (standardized) for downstream use.
The downstream multi-factor model consumer uses `dyld_score`.

## §12. STAGE 8 — Validation

### Required checks (all must pass before signing off)

**1. Standardization moments on ESTU:**
```
1a. max |cap-weighted mean of dyld_score|  < 1e-6     # machine-zero
1b. mean equal-weighted std of dyld_score ≈ 1.0 ± 0.02
```

**2. Raw descriptor sanity:**
```
2a. min(dyld) = 0.0 (no negative raw DYLD anywhere in the panel)
2b. median(dyld) in [0.000, 0.030]    # median pulled toward 0 by many non-payers
2c. max(dyld) after trim ≤ 0.30       # no extreme outlier surviving the 3σ clip
```

**3. Coverage stability:**
```
≥ 4,000 stocks with non-null dyld_score per date post-2005
```

**4. Factor stability (month-over-month rank correlation):**
```
MoM Spearman ρ > 0.90    # dividends change slowly; high persistence expected
```

**5. Economic direction (quintile monotonicity):**
```
Q5 mean dyld_score > Q1 mean dyld_score   (high-yield stocks in Q5, non-payers in Q1)
Q1 quintile should contain mostly zero-payers (dyld = 0)
```

**6. Disk vs memory equivalence:**
```
max abs diff of dyld values between memory and read-back parquet < 1e-10
```

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/`, your shared utilities.

## §13. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | TTM construction | 4-quarter rolling sum of ARQ DPS (datekey PIT) | Annual figure from ARY dimension | When ARY annual data is available in Sharadar |
| 2 | Zero-dividend handling | Retain `dyld = 0.0` (valid exposure for non-payers) | Exclude non-payers; compute only for dividend-paying stocks | If factor is used in a dividend-focused sub-model |
| 3 | Missing DPS (null) | Null `dyld` if n_obs=0; excluded from standardization | Treat null as 0 and include in standardization | If data quality improves and null reliably means "no dividend" |
| 4 | Negative DPS clamp | Clamp quarterly DPS to 0 before summing | Exclude quarter; reduce n_obs | Low impact; Sharadar rarely has negative DPS |
| 5 | PIT filter | `datekey ≤ signal_date` (filing date) | `calendardate ≤ signal_date` (fiscal quarter end) | Never — datekey is always correct for PIT |
| 6 | Minimum quarters | `MIN_QUARTERS = 1` | 2 or 4 quarters | If coverage is too broad and includes very thin-history names |
| 7 | Outlier trim | Standard 3σ; no hard yield cap | Hard cap at 20–25% yield before 3σ trim | If extreme REIT/MLP outliers persist after 3σ trim |

## §14. Common pitfalls — read this before coding

**Pitfall 1: Confusing zero dividend (valid) with missing DPS (invalid)**
`dps = 0.0` in Sharadar means the company paid no dividend that quarter — this is valid data. `dps = null` means the field was not reported. Treating null as 0 inflates coverage and produces spurious DYLD = 0 for companies that have no DPS history at all. Count `n_obs` using non-null `dps` values only.

**Pitfall 2: Division by zero on price**
SEP occasionally has zero or null close prices for delisted or suspended stocks. Guard `price > 0` before computing `dyld = ttm_dps / price`. Stocks failing this guard produce null `dyld`, not infinity.

**Pitfall 3: Using calendardate instead of datekey for PIT**
SF1 rows have both `calendardate` (fiscal quarter end) and `datekey` (actual filing date, always ≥ calendardate). Filtering on `calendardate ≤ signal_date` introduces look-ahead bias — a company's Q2 data (calendardate June 30) may not be filed until August 10 (datekey). Always PIT-filter on `datekey ≤ signal_date`.

**Pitfall 4: Rolling sum over wrong sort order**
The 4-quarter rolling sum must be computed after sorting rows by `datekey` within each `permaticker`. If rows are unsorted, `rolling_sum(...).over("permaticker")` will sum non-contiguous quarters and produce incorrect TTM values. Sort by `["permaticker", "datekey"]` before any rolling operation.

**Pitfall 5: Lower 3σ bound clipping zero-payers**
The lower 3σ bound `(cw_mean − 3·sigma)` will typically be negative. Applying `clip(lo, hi)` with `lo < 0` is harmless — DYLD ≥ 0 by construction so no values hit the lower bound. However, if you raise `lo = max(lo, 0)` to "clean up" the bound, verify this does not accidentally alter the clip logic for edge-case stocks with dyld near 0.

## §15. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/dyld_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3 exactly (`permaticker`, `signal_date`, `in_estu`, `mcap`, `dyld`, `dyld_score`, `n_obs`)
2. ✅ All 6 required validation checks in §12 pass within tolerance
3. ✅ ESTU has ~2,500–3,500 names per date, stable over time
4. ✅ Cap-weighted mean of `dyld_score` is machine-zero for every date in ESTU
5. ✅ Equal-weighted std of `dyld_score` is 1.0 (to 2 decimal places) for every date in ESTU
6. ✅ `min(dyld) = 0.0` and `median(dyld) ∈ [0.000, 0.030]` across dates
7. ✅ Coverage of `dyld_score` across coverage universe ≥ 4,000 stocks per date post-2005
8. ✅ Month-over-month rank stability ρ > 0.90
9. ✅ All ❓ NOT-IN-PDF decisions in §13 are documented somewhere in the code

**Once DYLD is built and passes validation, no downstream factors depend on it.**
DYLD is a standalone terminal factor in the USE4 style factor pipeline.